In [ ]:
# --- repo path bootstrap (added by the port) ---
import sys, pathlib
ROOT = next(p for p in pathlib.Path.cwd().parents if (p / "utils" / "paths.py").is_file())
sys.path.insert(0, str(ROOT))
from utils.paths import (profiles, features, feature_output, figdir, metadata,
                         data_dir, external, require)
from utils.panels import save_panel

import pandas as pd
import numpy as np
import os

# # Pycytominer
# from pycytominer import feature_select
# from pycytominer import normalize
# from pycytominer import aggregate

# Set current working directory


In [ ]:
cell_line = 'HT29' 

In [ ]:
exposed_wells = [
    f"{chr(r)}{c:02d}"
    for r in range(ord("B"), ord("O") + 1)
    for c in range(3, 23)
]



In [ ]:
def count_wells_per_plate(df: pd.DataFrame,
                          plate: str = 'Metadata_Barcode',
                          well: str = 'Metadata_Well',
                          count_col: str = 'counts') -> pd.DataFrame:
    return df.groupby(plate)[well] \
             .nunique() \
             .reset_index(name=count_col)

def load_features(dir: str,
                  cell_line: str = 'HT29'
                  ) -> pd.DataFrame:

    files = os.listdir(dir)

    # Select all files with HT29 in the name as well as MedianAgg_meanstd
    files = [file for file in files if cell_line in file and 'MedianAgg' in file]

    # Load all files
    data = []
    for file in files:
        data.append(pd.read_parquet(dir + file))
    data = pd.concat(data)
    
    return data

#### Load features from plate1 and perform post-hoc spheredetect

In [ ]:
# Load the CellProfiler features into a pandas dataframe

dataset_p1_loaded = load_features(dir=str(features("exp4_objective", "FeaturesImages_030625_CellPainting_20241220clearedspheroidsBOMI_20241220_151510".replace("FeaturesImages_", ""), "SingleSlice")) + "/",
                    cell_line=cell_line)

# Check that the data is complete
print(count_wells_per_plate(dataset_p1_loaded))

In [ ]:
# Load results from sphere detection

# Load the sphere detection output for plate 1
detected_df = pd.read_parquet(ROOT / "analysis" / "2_Processing" / "exp4_objective" / "data" / "sphere_detection_output_CellPainting_20241220clearedspheroidsBOMI_20241220_151510.parquet")

# Repopulate the Metadata_Barcode column with the correct barcode
detected_df['Metadata_Barcode'] = detected_df['image_path'].str.split(
    '/').str[5]

# Check that the data is complete
print(count_wells_per_plate(detected_df))


In [ ]:
# Detect spheroids post-hoc
dataset_p1 = dataset_p1_loaded.merge(detected_df[['Metadata_Well', 'Metadata_Z', 'Metadata_Z_calculated']], 
                     how='left', left_on=['Metadata_Well', 'Metadata_Site'], 
                     right_on=['Metadata_Well', 'Metadata_Z'])


# Filter spheroids within a range of Z values
dataset_p1 = dataset_p1.query('(Metadata_Z_calculated >= 0) & (Metadata_Z_calculated <= 18)')

# Check that the data is complete
print(count_wells_per_plate(dataset_p1))

In [ ]:
plate = "Before Christmas"

# List the wells that were "lost" so that I can inspect. 
wells_with_cells = list(sorted(dataset_p1_loaded.Metadata_Well.unique()))
wells_with_full_spheroids = list(sorted(dataset_p1.Metadata_Well.unique()))

# Display the wells that where exposed, but were no cells were detected. 
no_cells = [well for well in exposed_wells if well not in wells_with_cells]
print(f'No cells were detected in these wells: {no_cells}')

# Display the wells where no cells were detected, but a spheroid was detected 
no_cells = [well for well in wells_with_full_spheroids if well not in wells_with_cells]
print(f'No cells were detected in these wells: {no_cells}')


# Display the wells where cells were detected, but no spheroid was detected 
no_cells = [well for well in wells_with_cells if well not in wells_with_full_spheroids]
print(f'{plate}: No spheroids were detected in these wells: {no_cells}')


#### Load features from plate 2 and perform post-hoc spheredetect


In [ ]:
# Load the CellProfiler features into a pandas dataframe

dataset_p2_loaded = load_features(dir=str(features("exp4_objective", "FeaturesImages_030625_CellPainting_20250127Cellpaintcleared3D_20250127_171120".replace("FeaturesImages_", ""), "SingleSlice")) + "/",
                    cell_line=cell_line)

# Check that the data is complete
print(count_wells_per_plate(dataset_p2_loaded))

In [ ]:
# Load results from sphere detection

# Load the sphere detection output for plate 1
detected_df2 = pd.read_parquet(ROOT / "analysis" / "2_Processing" / "exp4_objective" / "data" / "sphere_detection_output_CellPainting_20250127Cellpaintcleared3D_20250127_171120.parquet")

# Repopulate the Metadata_Barcode column with the correct barcode
detected_df2['Metadata_Barcode'] = detected_df2['image_path'].str.split(
    '/').str[5]

# Check that the data is complete
print(count_wells_per_plate(detected_df2))


In [ ]:
# Detect spheroids post-hoc
dataset_p2 = dataset_p2_loaded.merge(detected_df2[['Metadata_Well', 'Metadata_Z', 'Metadata_Z_calculated']], 
                     how='left', left_on=['Metadata_Well', 'Metadata_Site'], 
                     right_on=['Metadata_Well', 'Metadata_Z'])


# Filter spheroids within a range of Z values
dataset_p2 = dataset_p2.query('(Metadata_Z_calculated >= 0) & (Metadata_Z_calculated <= 18)')

# Check that the data is complete
print(count_wells_per_plate(dataset_p2))

In [ ]:
plate = "After Christmas"

# List the wells that were "lost" so that I can inspect. 
wells_with_cells = list(sorted(dataset_p2_loaded.Metadata_Well.unique()))
wells_with_full_spheroids = list(sorted(dataset_p2.Metadata_Well.unique()))

# Display the wells that where exposed, but were no cells were detected. 
no_cells = [well for well in exposed_wells if well not in wells_with_cells]
print(f'No cells were detected in these wells: {no_cells}')

# Display the wells where no cells were detected, but a spheroid was detected 
no_cells = [well for well in wells_with_full_spheroids if well not in wells_with_cells]
print(f'No cells were detected in these wells: {no_cells}')


# Display the wells where cells were detected, but no spheroid was detected 
no_cells = [well for well in wells_with_cells if well not in wells_with_full_spheroids]
print(f'{plate}: No spheroids were detected in these wells: {no_cells}')


#### Plate 3
Idem dito

In [ ]:
# Load the CellProfiler features into a pandas dataframe

dataset_p3_loaded = load_features(dir=str(features("exp4_objective", "FeaturesImages_030625_CellPainting_CellPaint3DBomi_WI_for_Jordi_20250203_155142".replace("FeaturesImages_", ""), "SingleSlice")) + "/",
                    cell_line=cell_line)

# Check that the data is complete
print(count_wells_per_plate(dataset_p3_loaded))

In [ ]:
# Load results from sphere detection

# Load the sphere detection output for plate 1
detected_df3 = pd.read_parquet(ROOT / "analysis" / "2_Processing" / "exp4_objective" / "data" / "sphere_detection_output_CellPainting_CellPaint3DBomi_WI_for_Jordi_20250203_155142.parquet")

# Repopulate the Metadata_Barcode column with the correct barcode
detected_df3['Metadata_Barcode'] = detected_df3['image_path'].str.split(
    '/').str[5]

# Check that the data is complete
print(count_wells_per_plate(detected_df3))

In [ ]:
# Detect spheroids post-hoc
dataset_p3 = dataset_p3_loaded.merge(detected_df3[['Metadata_Well', 'Metadata_Z', 'Metadata_Z_calculated']],
                     how='left', left_on=['Metadata_Well', 'Metadata_Site'],
                     right_on=['Metadata_Well', 'Metadata_Z'])


# Filter spheroids within a range of Z values
dataset_p3 = dataset_p3.query('(Metadata_Z_calculated >= 0) & (Metadata_Z_calculated <= 18)')

# Check that the data is complete
print(count_wells_per_plate(dataset_p3))

In [ ]:
plate = "WI"

# List the wells that were "lost" so that I can inspect. 
wells_with_cells = list(sorted(dataset_p3_loaded.Metadata_Well.unique()))
wells_with_full_spheroids = list(sorted(dataset_p3.Metadata_Well.unique()))

# Display the wells that where exposed, but were no cells were detected. 
no_cells = [well for well in exposed_wells if well not in wells_with_cells]
print(f'No cells were detected in these wells: {no_cells}')

# Display the wells where no cells were detected, but a spheroid was detected 
no_cells = [well for well in wells_with_full_spheroids if well not in wells_with_cells]
print(f'No cells were detected in these wells: {no_cells}')


# Display the wells where cells were detected, but no spheroid was detected 
no_cells = [well for well in wells_with_cells if well not in wells_with_full_spheroids]
print(f'{plate}: No spheroids were detected in these wells: {no_cells}')


#### Save the data in a single parquet file

In [ ]:
# Save the data
OutputDir = str(profiles("exp4_objective", "")) + "/"
if not os.path.exists(OutputDir): 
    os.makedirs(OutputDir)


# Concatenate the datasets
dataset_complete = pd.concat([dataset_p1, dataset_p2, dataset_p3], axis=0)    

# Save as parquet
dataset_complete.to_parquet(('{}{}_030625_SliceMedianAgg_Combined.parquet').format(OutputDir, cell_line))

In [ ]:
# What is the count of planes per well?
planes_per_well = dataset_complete.groupby(['Metadata_Barcode', 'Metadata_Well']).size().reset_index(name='counts')
planes_per_well

# Very few wells have fewer than 19 planes.